# Ejecutor Colab — Pregunta 3 (YOLO, CCT-20)
Runtime → GPU (T4). Este cuaderno solo clona el repo y llama a los scripts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK='/content/drive/MyDrive/cct_yolo'   # pesos y resultados persisten aqui
!mkdir -p $WORK

In [ ]:
REPO_URL='https://github.com/USUARIO/camera-traps-yolo.git'  # <-- cambiar
!git clone $REPO_URL /content/repo || (cd /content/repo && git pull)
%cd /content/repo
!pip install -q -r requirements.txt

## Datos
Descarga las imágenes reducidas y las anotaciones desde LILA BC (https://lila.science/datasets/caltech-camera-traps) **a Drive una sola vez**, y apunta las rutas. Verifica los nombres exactos de los archivos en la página.

In [ ]:
IMAGES_ROOT='/content/drive/MyDrive/cct_data/eccv_18_all_images_sm'   # <-- ajustar
ANN_DIR='/content/drive/MyDrive/cct_data/eccv_18_annotation_files'        # <-- ajustar
!ls $IMAGES_ROOT | head -3; ls $ANN_DIR

In [ ]:
import os
os.environ.update(IMAGES_ROOT=IMAGES_ROOT, ANN_DIR=ANN_DIR, MODEL='yolov8s.pt', EPOCHS='30', IMGSZ='640')
!python src/prepare_data.py --images-root $IMAGES_ROOT --train-json $ANN_DIR/train_annotations.json --cis-val-json $ANN_DIR/cis_val_annotations.json --trans-val-json $ANN_DIR/trans_val_annotations.json --max-empty-ratio 1.0

## Entrenamiento (prueba rápida con 3 épocas antes de lanzar la corrida completa)

In [ ]:
!python src/train.py --model $MODEL --epochs 3 --imgsz $IMGSZ

In [ ]:
!python src/train.py --model $MODEL --epochs $EPOCHS --imgsz $IMGSZ
!cp -r runs $WORK/

## Evaluación y análisis

In [ ]:
W='runs/cct_yolo/weights/best.pt'
!python src/evaluate.py --weights $W
!python src/robustness.py --weights $W
!python src/gradcam_analysis.py --weights $W
!python src/monitoring.py --weights $W
!cp -r results $WORK/

## Subir resultados al repo
Descarga `results/` (tablas y figuras) y haz commit para el informe. No subas pesos ni datos.